# Notebook 5 · Ajuste final e previsões para o Kaggle

O NB_4 escolheu a receita (algoritmo, seleção de variáveis e hiperparâmetros) usando apenas divisões temporais internas. Neste notebook:

1. reconstruímos a mesma preparação dos NB_2/3/4 e carregamos a receita guardada em `output/modelling/receita_escolhida.json`;
2. verificamos que a reconstrução reproduz o resultado do NB_4 (controlo de reprodutibilidade);
3. **reaprendemos tudo (preparação, seleção e modelo) nas 6469 compras utilizáveis de `train.csv`**, incluindo as linhas que o NB_4 tinha deixado na margem de 90 dias;
4. geramos previsões para as 986 compras de `test.csv`, mantendo o `ID` e a ordem de `sample_submission.csv`;
5. validamos o formato e guardamos o ficheiro de submissão em `data/Processed/`.

`test.csv` só é usado no passo 4, para `predict_proba`: nenhuma transformação, seleção ou hiperparâmetro é ajustado com ele.

## 1. Dados e limpeza

Repetimos exatamente a limpeza do NB_4 (as mesmas regras aplicam-se a treino e teste, mas são regras fixas, sem parâmetros aprendidos):

- `prior_mean_purchase_value` inconsistente é reconstruída como `prior_total_spend / prior_purchase_count` quando os componentes o permitem; caso contrário fica ausente;
- `spend_365d` inválido (negativo, absurdo ou superior ao gasto histórico) e intervalos negativos passam a ausentes;
- as 65 linhas de treino sem qualquer informação de compra são excluídas (o teste mantém todas as 986 linhas).

In [ ]:
import hashlib, json, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data" / "Raw" / "train.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "Raw"
assert (DATA / "train.csv").exists(), "Executa a partir da raiz ou de notebooks/."

TARGET = "repeat_purchase_90d"
RANDOM_STATE = 42

train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
SAMPLE_HASH = hashlib.md5((DATA / "sample_submission.csv").read_bytes()).hexdigest()  # para provar que não o alteramos

assert TARGET in train_raw and TARGET not in test_raw
assert set(test_raw.columns) == set(train_raw.columns) - {TARGET}
assert train_raw["ID"].is_unique and test_raw["ID"].is_unique
assert sample.columns.tolist() == ["ID", TARGET]
assert sample["ID"].tolist() == test_raw["ID"].tolist()
print({"train": train_raw.shape, "test": test_raw.shape, "sample": sample.shape})
print({"python": platform.python_version(), "scikit-learn": sklearn.__version__,
       "pandas": pd.__version__, "numpy": np.__version__})

In [ ]:
def clean_fixed(frame):
    out = frame.copy()
    out["purchase_date"] = pd.to_datetime(out["purchase_date"], errors="coerce")

    col = "prior_mean_purchase_value"
    invalid_mean = out[col].lt(0) | out[col].ge(1e8)
    valid_parts = out["prior_purchase_count"].gt(0) & out["prior_total_spend"].ge(0)
    reconstructed = out["prior_total_spend"] / out["prior_purchase_count"].replace(0, np.nan)
    out.loc[invalid_mean & valid_parts, col] = reconstructed[invalid_mean & valid_parts]
    out.loc[invalid_mean & ~valid_parts, col] = np.nan

    outside_bounds = (
        out[col].lt(out["prior_min_purchase_value"] - 0.05) |
        out[col].gt(out["prior_max_purchase_value"] + 0.05)
    )
    reconstructed_in_bounds = (
        reconstructed.ge(out["prior_min_purchase_value"] - 0.05) &
        reconstructed.le(out["prior_max_purchase_value"] + 0.05)
    )
    repair_bounds = outside_bounds & reconstructed_in_bounds & valid_parts
    out.loc[repair_bounds, col] = reconstructed[repair_bounds]

    identity_disagrees = (out[col] - reconstructed).abs().gt(
        np.maximum(0.05, 0.01 * reconstructed.abs())
    )
    repair_identity = identity_disagrees & reconstructed_in_bounds & valid_parts
    out.loc[repair_identity, col] = reconstructed[repair_identity]

    invalid_spend = (
        out["spend_365d"].lt(0) |
        out["spend_365d"].ge(1e8) |
        out["spend_365d"].gt(out["prior_total_spend"] + 0.05)
    )
    out["spend_365d"] = out["spend_365d"].mask(invalid_spend)

    for gap in ("days_since_previous_purchase", "prior_mean_gap_days"):
        out[gap] = out[gap].mask(out[gap].lt(0))
    return out

train_clean = clean_fixed(train_raw)
test_clean = clean_fixed(test_raw)

purchase_totals = ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_", "Custo"]
no_purchase_info = train_clean[purchase_totals].isna().all(axis=1)
usable = ~no_purchase_info & train_clean["purchase_date"].notna() & train_clean[TARGET].notna()
print("Linhas de treino utilizáveis:", int(usable.sum()), "| excluídas:", int((~usable).sum()))
assert test_clean["purchase_date"].notna().all(), "Datas inválidas no teste"
assert len(test_clean) == len(sample)

## 2. Variáveis de base

Mantemos as 34 variáveis de base do NB_4. Ficam fora: identificadores (`ID`, `customer_id`), a data absoluta, o alvo e `random_noise`. A ficha `clientes.csv` não é integrada (decisão dos NB_2/3, confirmada no NB_4).

In [ ]:
def make_features(clean_frame):
    X = clean_frame.drop(columns=["ID", "customer_id", "purchase_date", TARGET, "random_noise"],
                         errors="ignore")
    text_cols = X.select_dtypes(include=["str", "object", "category"]).columns
    for col in text_cols:
        X[col] = X[col].astype(object).where(X[col].notna(), np.nan)
    return X

X_all = make_features(train_clean)
X_test = make_features(test_clean)
X = X_all.loc[usable].copy()
y = train_clean.loc[usable, TARGET].astype(int)
dates = train_clean.loc[usable, "purchase_date"]

assert X.columns.tolist() == X_test.columns.tolist()
assert len(X_test) == len(test_raw)
print("Variáveis de base:", X.shape[1], "| taxa de recompra no treino:", round(y.mean(), 4))

## 3. Preparação, seleção e modelo (receita do NB_4)

A preparação (imputação por mediana com indicadores de ausência, categoria `__missing__`, one-hot com agrupamento de categorias raras, padronização, remoção de constantes) e a eventual seleção L1 fazem parte do `Pipeline`. Assim são **aprendidas apenas com os dados a que o pipeline é ajustado**: no ajuste final, o treino completo; nunca o teste.

A receita é lida do JSON do NB_4. Se o ficheiro não existir, usamos a receita registada no NB_4 (Random Forest, todas as colunas, `max_depth=12`, `min_samples_leaf=5`).

In [ ]:
def make_preprocessor(X_reference, scale_numeric=True):
    numeric_cols = X_reference.select_dtypes(include="number").columns.tolist()
    category_cols = X_reference.select_dtypes(exclude="number").columns.tolist()
    numeric_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True,
                                              keep_empty_features=True))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    category_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__missing__",
                                 keep_empty_features=True)),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20)),
    ])
    return ColumnTransformer([("numeric", Pipeline(numeric_steps), numeric_cols),
                              ("category", category_pipe, category_cols)])

MODELS = {
    "Logística": LogisticRegression(solver="liblinear", C=1, max_iter=3000, random_state=RANDOM_STATE),
    "Árvore": DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                             n_jobs=1, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=31, weights="distance", n_jobs=1),
    "SVM RBF": SVC(C=1, gamma="scale", probability=True, cache_size=256, random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
}

def build_model_pipeline(X_reference, model_name, selection, model_params=None):
    preprocessor = make_preprocessor(X_reference, scale_numeric=True)
    preprocessor.set_params(sparse_threshold=0)
    selector = "passthrough"
    if selection == "L1 C=0.1":
        selector = SelectFromModel(
            LogisticRegression(solver="liblinear", l1_ratio=1, C=0.1, max_iter=3000,
                               random_state=RANDOM_STATE),
            threshold=1e-8)
    else:
        assert selection == "Todas"
    model = clone(MODELS[model_name])
    if model_params:
        model.set_params(**model_params)
    return Pipeline([("preprocess", preprocessor), ("variance", VarianceThreshold()),
                     ("select", selector), ("model", model)])

def selected_feature_names(pipeline):
    names = pipeline.named_steps["preprocess"].get_feature_names_out()
    names = names[pipeline.named_steps["variance"].get_support()]
    selector = pipeline.named_steps["select"]
    if not isinstance(selector, str):
        names = names[selector.get_support()]
    return list(names)

RECIPE_PATH = ROOT / "output" / "modelling" / "receita_escolhida.json"
FALLBACK = {"modelo": "Random forest", "seleção": "Todas",
            "parâmetros": {"max_depth": 12, "min_samples_leaf": 5}}
if RECIPE_PATH.exists():
    recipe = json.loads(RECIPE_PATH.read_text(encoding="utf-8"))
    print("Receita carregada de", RECIPE_PATH.name)
else:
    recipe = FALLBACK
    print("AVISO: receita_escolhida.json não encontrada; a usar a receita registada no NB_4.")

model_name, selection, params = recipe["modelo"], recipe["seleção"], recipe["parâmetros"]
print({"modelo": model_name, "seleção": selection, "parâmetros": params})
if "versoes" in recipe and recipe["versoes"].get("sklearn") != sklearn.__version__:
    print("AVISO: versão do scikit-learn diferente da do NB_4:", recipe["versoes"].get("sklearn"))

## 4. Controlo de reprodutibilidade

Antes do ajuste final, confirmamos que esta reconstrução é fiel ao NB_4: repetimos a divisão temporal (80 % / margem de 90 dias), ajustamos em `X_fit` e comparamos a ROC-AUC em `X_val` com o valor guardado na receita (0.8016 no NB_4). Este passo é apenas um teste de integridade do código; não altera nenhuma escolha.

In [ ]:
boundary = dates.sort_values().iloc[int(0.8 * len(dates))]
purge_start = boundary - pd.Timedelta(days=90)
fit_rows, val_rows = dates < purge_start, dates >= boundary
assert dates.loc[fit_rows].max() < purge_start and dates.loc[val_rows].min() >= boundary
X_fit, y_fit, X_val, y_val = X.loc[fit_rows], y.loc[fit_rows], X.loc[val_rows], y.loc[val_rows]
print("Ajuste:", len(X_fit), "| validação:", len(X_val), "| margem:", int((~fit_rows & ~val_rows).sum()))

check_pipeline = build_model_pipeline(X_fit, model_name, selection, params)
check_pipeline.fit(X_fit, y_fit)
auc_check = roc_auc_score(y_val, check_pipeline.predict_proba(X_val)[:, 1])
auc_saved = recipe.get("auc_Xval")
print(f"ROC-AUC em X_val (reconstrução): {auc_check:.4f}" +
      (f" | guardada no NB_4: {auc_saved:.4f}" if auc_saved else ""))
if auc_saved is not None and abs(auc_check - auc_saved) > 0.005:
    print("AVISO: diferença face ao NB_4; confirmar versões das bibliotecas e dados.")

## 5. Ajuste final em todo o treino

A estimativa de desempenho esperado é a do NB_4 (AUC média interna ≈ 0.780; AUC ≈ 0.80 no período `X_val`, que não é totalmente independente). Para as previsões finais, usamos **todas as 6469 compras utilizáveis**: mais dados e, sobretudo, os períodos mais recentes, que são os mais próximos do teste. Preparação, seleção e modelo são reaprendidos de raiz.

In [ ]:
final_pipeline = build_model_pipeline(X, model_name, selection, params)
final_pipeline.fit(X, y)
final_names = selected_feature_names(final_pipeline)
print("Compras usadas no ajuste final:", len(X), "| colunas após preparação e seleção:", len(final_names))

rf = final_pipeline.named_steps["model"]
if hasattr(rf, "feature_importances_"):
    kept = final_names
    imp = pd.Series(rf.feature_importances_, index=kept).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(8, 5))
    imp.sort_values().plot.barh(ax=ax, color="#1779A8")
    ax.set(title="Top 15 importâncias (modelo final)", xlabel="Importância")
    fig.tight_layout(); plt.show()

## 6. Previsões para `test.csv`

O enunciado pede previsões para **todas** as observações de `test.csv`, com o `ID` inalterado, no formato do `sample_submission.csv`. Geramos a probabilidade de recompra de cada compra e produzimos **um único ficheiro** com as colunas `ID` e `repeat_purchase_90d`. Se a coluna do alvo no `sample_submission.csv` tiver apenas 0/1, a submissão leva classes (limiar 0.5); caso contrário, probabilidades. O ficheiro é guardado em `data/Processed/sample_submission.csv`; o original em `data/Raw/` não é modificado.

In [ ]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    test_proba = final_pipeline.predict_proba(X_test)[:, 1]
for item in {(x.category.__name__, str(x.message)) for x in w}:
    print("Aviso:", item)

assert len(test_proba) == len(test_raw) == len(sample)
assert np.isfinite(test_proba).all() and ((test_proba >= 0) & (test_proba <= 1)).all()

THRESHOLD = 0.5
sample_is_binary = set(sample[TARGET].dropna().unique()) <= {0, 1}
values = (test_proba >= THRESHOLD).astype(int) if sample_is_binary else test_proba
submission = pd.DataFrame({"ID": test_raw["ID"].values, TARGET: values})
print("Formato inferido do sample_submission:", "classes 0/1" if sample_is_binary else "probabilidades")
print("Taxa média prevista (teste):", round(test_proba.mean(), 4),
      "| taxa de recompra no treino:", round(y.mean(), 4),
      "| % previsto como 1 (limiar 0.5):", round((test_proba >= THRESHOLD).mean(), 4))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(test_proba, bins=30, color="#1779A8")
ax.axvline(THRESHOLD, ls="--", color="grey")
ax.set(title="Distribuição das probabilidades previstas no teste", xlabel="P(repeat_purchase_90d = 1)")
fig.tight_layout(); plt.show()

## 7. Validação e guardar a submissão

Verificações finais: mesmas colunas, mesmo número de linhas e mesma ordem de IDs que o `sample_submission.csv`, sem ausentes, sem duplicados e uma linha por compra de teste.

In [ ]:
assert submission.columns.tolist() == sample.columns.tolist()
assert submission["ID"].tolist() == sample["ID"].tolist()
assert submission["ID"].is_unique and submission.shape == sample.shape
assert submission[TARGET].notna().all()

OUT = ROOT / "data" / "Processed"
OUT.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUT / "sample_submission.csv", index=False)  # mesmo formato do ficheiro original, que fica intacto em Raw
assert hashlib.md5((DATA / "sample_submission.csv").read_bytes()).hexdigest() == SAMPLE_HASH
print("Raw/sample_submission.csv intacto; submissão guardada em", OUT / "sample_submission.csv")
display(submission.head())

## 8. Resumo e limitações

- **Pipeline final:** limpeza fixa → variáveis de base (34, sem identificadores, data, alvo nem `random_noise`) → imputação/indicadores/one-hot/padronização → Random Forest (receita do NB_4) ajustada em todas as compras utilizáveis.
- **Sem fuga de informação:** transformações e modelo aprendidos só com `train.csv`; o teste só passa por `predict_proba`.
- **Desempenho esperado:** ROC-AUC ≈ 0.78 nas divisões internas e ≈ 0.80 em `X_val` (diagnóstico, não totalmente independente); o resultado no Kaggle pode diferir.
- **Limitações:** hiperparâmetros escolhidos com poucas divisões temporais reutilizadas (viés de escolha); sem calibração nem otimização do limiar; ficha de clientes não utilizada.

### Fim do percurso

Este é o último notebook da sequência. O ficheiro de previsões gerado constitui o resultado final para submissão no Kaggle; não existe um notebook seguinte.
